# register-buffer — ex1: register non-trainable state and verify state_dict membership

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `register-buffer`. Running the final beacon cell reports progress against the `PyTorch: register_buffer` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: register_buffer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-buffer`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-buffer"
DD_SUBTOPIC = "PyTorch: register_buffer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `register_buffer` — quick refresher

A **buffer** is non-trainable state that belongs to a Module — it's saved in `state_dict`, moved by `.to(device)`, but NOT updated by the optimizer. The idiom:

```
class MyBN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.weight = nn.Parameter(t.ones(num_features))     # trainable
        self.bias   = nn.Parameter(t.zeros(num_features))    # trainable
        self.register_buffer('running_mean', t.zeros(num_features))
        self.register_buffer('running_var',  t.ones(num_features))
```

**Buffer vs Parameter — what differs:**

| | `nn.Parameter` | `register_buffer` |
|--|--|--|
| Appears in `.parameters()` | yes | NO |
| Appears in `.buffers()` | no | yes |
| Appears in `state_dict` | yes | yes |
| Follows `.to(device)` | yes | yes |
| `requires_grad` defaults | True | False (no grad) |
| Updated by optimizer | yes | NO |

**Why buffers exist.** Sometimes a Module needs to remember state that's NOT learned — BatchNorm's running mean/var, attention masks, positional-encoding tables, scaling constants. Marking them as buffers gets them saved/loaded/moved automatically without letting the optimizer touch them.

**Critical distinction from plain `self.tensor = ...`.** A plain tensor attribute (`self.running_mean = t.zeros(C)`) does NOT appear in `state_dict`, does NOT follow `.to(device)`, and gets silently broken when the model is saved/restored. Always use `register_buffer` for non-trainable tensor state.

**Access pattern.** After registration, you read/write via `self.running_mean` like any attribute — PyTorch installs a descriptor that does the right thing under the hood.

### Exercise 1 — register non-trainable state and verify state_dict membership

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `self.register_buffer(name, tensor)` to attach non-trainable state to a Module, and discriminate it from `nn.Parameter` via `.parameters()`, `.buffers()`, and `state_dict` membership.
> Keywords: register_buffer, state_dict, non-trainable, buffers
> ```

**KCs targeted:** `register-buffer-syntax`, `buffer-vs-parameter-discrimination`

Implement a `MyBN` Module that mimics the **state shape** of `nn.BatchNorm2d` — without doing any actual normalization. The module has FOUR pieces of channel-wise state:

- `weight`: `nn.Parameter` of shape `(num_features,)`, initialized to **ones**. **Trainable.**
- `bias`:   `nn.Parameter` of shape `(num_features,)`, initialized to **zeros**. **Trainable.**
- `running_mean`: buffer of shape `(num_features,)`, initialized to **zeros**. **NOT trainable** — registered via `self.register_buffer('running_mean', t.zeros(...))`.
- `running_var`:  buffer of shape `(num_features,)`, initialized to **ones**. **NOT trainable** — registered via `self.register_buffer('running_var', t.ones(...))`.

No forward method needed — this drill is about getting the STATE registration right, not the math (the math is in the `batchnorm-running-stats` and `batchnorm-affine-params` drills).

Skeleton:

```
class MyBN(nn.Module):
    def __init__(self, num_features: int):
        super().__init__()
        # self.weight = nn.Parameter(...)
        # self.bias   = nn.Parameter(...)
        # self.register_buffer('running_mean', ...)
        # self.register_buffer('running_var',  ...)
```

**What the test verifies (exercising the discrimination skill).**
- `weight` and `bias` show up in `.parameters()`, NOT in `.buffers()`.
- `running_mean` and `running_var` show up in `.buffers()`, NOT in `.parameters()`.
- ALL FOUR show up in `state_dict()`.
- Buffers have `requires_grad == False`.
- Calling `.to('cpu')` (no-op on CPU but exercises the path) preserves all four.
- A round-trip `state_dict()` → `load_state_dict()` preserves the running stats verbatim.

In [ ]:
import torch.nn as nn

class MyBN(nn.Module):
    def __init__(self, num_features: int):
        super().__init__()
        raise NotImplementedError()


def _test_ex1():
    import torch.nn as nn

    C = 8
    bn = MyBN(C)

    # All four attributes exist and have correct shape & init.
    assert hasattr(bn, 'weight') and bn.weight.shape == (C,)
    assert hasattr(bn, 'bias')   and bn.bias.shape == (C,)
    assert hasattr(bn, 'running_mean') and bn.running_mean.shape == (C,)
    assert hasattr(bn, 'running_var')  and bn.running_var.shape == (C,)
    assert t.allclose(bn.weight,       t.ones(C)),  'weight should init to ones'
    assert t.allclose(bn.bias,         t.zeros(C)), 'bias should init to zeros'
    assert t.allclose(bn.running_mean, t.zeros(C)), 'running_mean should init to zeros'
    assert t.allclose(bn.running_var,  t.ones(C)),  'running_var should init to ones'

    # weight/bias are Parameters; running_* are NOT.
    assert isinstance(bn.weight, nn.Parameter), 'weight must be nn.Parameter'
    assert isinstance(bn.bias,   nn.Parameter), 'bias must be nn.Parameter'
    assert not isinstance(bn.running_mean, nn.Parameter), 'running_mean must NOT be nn.Parameter'
    assert not isinstance(bn.running_var,  nn.Parameter), 'running_var must NOT be nn.Parameter'

    # Parameter / buffer membership discrimination.
    param_names = {n for n, _ in bn.named_parameters()}
    buf_names   = {n for n, _ in bn.named_buffers()}
    assert param_names == {'weight', 'bias'}, (
        f'expected params={{weight, bias}}, got {param_names}'
    )
    assert buf_names == {'running_mean', 'running_var'}, (
        f'expected buffers={{running_mean, running_var}}, got {buf_names}'
    )

    # state_dict has ALL FOUR (this is the whole reason buffers exist).
    sd = bn.state_dict()
    assert set(sd.keys()) == {'weight', 'bias', 'running_mean', 'running_var'}, (
        f'state_dict keys wrong: {set(sd.keys())}'
    )

    # requires_grad: True for params, False for buffers.
    assert bn.weight.requires_grad is True
    assert bn.bias.requires_grad   is True
    assert bn.running_mean.requires_grad is False, 'buffers should not require grad'
    assert bn.running_var.requires_grad  is False, 'buffers should not require grad'

    # .to('cpu') is a no-op on CPU but must not lose buffers (often a bug source).
    bn2 = bn.to('cpu')
    assert bn2.running_mean is not None and bn2.running_var is not None
    assert t.allclose(bn2.running_mean, t.zeros(C))

    # Round-trip: save, mutate, load, confirm restored.
    bn.running_mean.data.fill_(42.0)
    bn.running_var.data.fill_(3.14)
    sd2 = bn.state_dict()
    bn_fresh = MyBN(C)
    bn_fresh.load_state_dict(sd2)
    assert t.allclose(bn_fresh.running_mean, t.full((C,), 42.0)), 'running_mean lost in save/load round-trip'
    assert t.allclose(bn_fresh.running_var,  t.full((C,), 3.14)), 'running_var lost in save/load round-trip'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
class MyBN(nn.Module):
    def __init__(self, num_features: int):
        super().__init__()
        self.weight = nn.Parameter(t.ones(num_features))
        self.bias   = nn.Parameter(t.zeros(num_features))
        self.register_buffer('running_mean', t.zeros(num_features))
        self.register_buffer('running_var',  t.ones(num_features))
```

**Why `register_buffer` (not `self.running_mean = t.zeros(...)`).** A plain tensor attribute is invisible to PyTorch's machinery — it doesn't appear in `state_dict`, doesn't follow `.to(device)`, and gets lost when the model is pickled / re-instantiated. Without registration the running stats would be silently broken across save/load cycles. THIS is the single most-common 'why does my model give different outputs after I save and reload it?' bug.

**Why buffers don't need a `nn.Buffer` wrapper.** Unlike parameters (which use `nn.Parameter` to mark them), buffers are tracked by NAME via the registration call. PyTorch stores them in a separate `_buffers` ordered dict on the module; that's the source of truth for `.buffers()` and the state_dict layout.

**`persistent=True` is the default.** `register_buffer(name, tensor, persistent=False)` exists for buffers you want at runtime but NOT in `state_dict` — e.g., a cached positional-encoding table that can be regenerated. Rarely needed; default `persistent=True` is the right call for running stats.

**Real BatchNorm has a third buffer.** `nn.BatchNorm2d` also registers `num_batches_tracked` (a long-tensor counter for stale-stat warnings). Out of scope for this drill, but illustrates the pattern: ANY non-trainable state that needs to survive save/load goes through `register_buffer`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()